# E80 LiveCodeBench 175문항

소스·데이터 ZIP을 `MyDrive/Putare`에, 모델을 `MyDrive/models`에 별도 업로드하세요. Drive를 직접 마운트하고 필요한 단계만 켜세요. 기본 Run all은 설치·추론·채점을 실행하지 않습니다.

pruning에 관측한 100문제를 포함한 175문항이므로 독립 held-out 평가는 아닙니다. context 262144에서 Q8 KV 약 29.22 GiB, 모델 파일과 합계 약 38.03 GiB는 작업 버퍼를 제외한 추정입니다. A100 80GB 또는 Blackwell 96GB를 권장하며 40GB는 여유가 부족할 수 있습니다. 실제 CUDA 동작은 미검증입니다.

모든 처리 토큰과 30개 층의 선택 Expert 8개 ID와 정규화 gate 가중치를 보존하며 weighted_output_norms는 계산하거나 저장하지 않습니다. 각 문제의 전체 routing 원본은 검증된 gzip으로 보존하며 크기가 매우 커질 수 있습니다. 결과는 런타임 디스크에 저장됩니다. CHECKPOINT_DIR에 Drive 경로를 지정하면 각 완료 문제를 검증 후 백업하며, 저장 공간이 부족하면 실행이 중단됩니다. 기본값 None은 백업하지 않습니다. 계측 callback 경계에서는 일부 CUDA fusion이 제한될 수 있습니다. 공식 채점은 비격리 코드 실행입니다.


In [5]:
SETUP_COLAB = True
RUN_GENERATION = True
RUN_GRADING = True
SOURCE_ZIP = '/content/drive/MyDrive/Putare/putare07-source-6b061e47d30a.zip'
DATA_ZIP = '/content/drive/MyDrive/Putare/putare07-data-0a3dd9112ddf.zip'
MODEL_PATH = '/content/drive/MyDrive/models/gemma-4-26B-A4B-it-UD-IQ4_NL-lcb100-activity-ordered-e80.gguf'

if SETUP_COLAB:
    _PUTARE_SETUP_READY = False
    from google.colab import drive
    drive.mount('/content/drive')

CHECKPOINT_DIR = None


Mounted at /content/drive


In [6]:
import hashlib, json, os, shutil, sys, zipfile
from pathlib import Path, PurePosixPath

def sha256_file(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def restore_archives(root, archives):
    """모든 ZIP과 기존 경로를 먼저 검증하고 누락된 파일만 복원한다."""
    # 2026-11-09: 불완전한 전송과 경로 탈출·기존 데이터 덮어쓰기를 함께 방지한다.
    root = Path(root)
    if root.is_symlink():
        raise ValueError('복원 위치가 심볼릭 링크입니다')
    validated = []
    used = set()
    try:
        for archive_path, expected_hash in archives:
            archive_path = Path(archive_path)
            if not archive_path.is_file():
                raise FileNotFoundError(f'Drive에 ZIP을 업로드하세요: {archive_path}')
            if sha256_file(archive_path) != expected_hash:
                raise ValueError(f'ZIP SHA256 불일치: {archive_path}')
            archive = zipfile.ZipFile(archive_path)
            validated.append((archive, {}))
            names = archive.namelist()
            if len(names) != len(set(names)) or '_putare_inventory.json' not in names:
                raise ValueError('ZIP 파일 목록에 중복 또는 인벤토리 누락')
            inventory = json.loads(archive.read('_putare_inventory.json'))
            if not isinstance(inventory, dict) or set(names) != set(inventory) | {'_putare_inventory.json'}:
                raise ValueError('ZIP 인벤토리 불일치')
            for name, digest in inventory.items():
                path = PurePosixPath(name)
                if (not isinstance(name, str) or path.is_absolute() or not path.parts
                        or any(p in ('', '.', '..') for p in name.split('/')) or '\\' in name or ':' in name):
                    raise ValueError('안전하지 않은 ZIP 경로')
                info = archive.getinfo(name)
                if info.is_dir() or (info.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError('ZIP에 링크 또는 디렉터리가 있습니다')
                if name in used:
                    raise ValueError('소스와 데이터 ZIP에 중복 경로가 있습니다: ' + name)
                used.add(name)
                with archive.open(info) as stream:
                    if hashlib.file_digest(stream, 'sha256').hexdigest() != digest:
                        raise ValueError('ZIP 파일 SHA256 불일치: ' + name)
                target = root.joinpath(*path.parts)
                for part in [root, *[root.joinpath(*path.parts[:i]) for i in range(1, len(path.parts) + 1)]]:
                    if part.is_symlink():
                        raise ValueError('복원 경로에 심볼릭 링크가 있습니다: ' + str(part))
                    if part != target and part.exists() and not part.is_dir():
                        raise FileExistsError('복원 폴더 위치가 파일입니다: ' + str(part))
                if target.exists() and (not target.is_file() or sha256_file(target) != digest):
                    raise FileExistsError('기존 파일이 달라 덮어쓰지 않습니다: ' + str(target))
                validated[-1][1][name] = target
        for archive, paths in validated:
            for name, target in paths.items():
                if not target.exists():
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with archive.open(name) as stream, target.open('xb') as output:
                        shutil.copyfileobj(stream, output)
    except (zipfile.BadZipFile, KeyError, UnicodeError) as error:
        raise ValueError('ZIP이 손상되었습니다') from error
    finally:
        for archive, _ in validated:
            archive.close()
    return len(used)


if SETUP_COLAB:
    _PUTARE_SETUP_READY = False
    _source_sha256 = '6b061e47d30ad9b9c9f13637d5f211e47f9f509b70307c2d592106f652d57d97'
    _data_sha256 = '0a3dd9112ddfed70ddab7afcee7f3dae57759180a152a691175b035af5f95e7f'
    ROOT = Path('/content/Putare') / (_source_sha256[:12] + '-' + _data_sha256[:12])
    if not (3, 12) <= sys.version_info[:2] < (3, 13):
        raise RuntimeError('Python 3.12가 필요합니다. Colab 런타임 버전을 확인하세요.')
    if not os.path.ismount('/content/drive'):
        raise RuntimeError('Google Drive가 마운트되지 않았습니다. 설정 셀에서 직접 마운트하세요.')
    # 2026-11-09: 같은 커널의 이전 버전 모듈 혼용을 막는다.
    for _name, _module in list(sys.modules.items()):
        if _name not in ('tools', 'moe_prune') and not _name.startswith(('tools.', 'moe_prune.')):
            continue
        _file = getattr(_module, '__file__', None)
        if _file and not Path(_file).resolve().is_relative_to(ROOT.resolve()):
            raise RuntimeError('다른 Putare 버전이 이미 로드되었습니다. 커널을 다시 시작하세요: ' + _name)
        _paths = getattr(_module, '__path__', ()) if not _file else ()
        if any(not Path(p).resolve().is_relative_to(ROOT.resolve()) for p in _paths):
            raise RuntimeError('다른 Putare 버전이 이미 로드되었습니다. 커널을 다시 시작하세요: ' + _name)
    restore_archives(ROOT, [(SOURCE_ZIP, _source_sha256), (DATA_ZIP, _data_sha256)])
    os.chdir(ROOT)
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    from tools.colab_setup import prepare_environment
    _PUTARE_SETUP_STATE = prepare_environment(ROOT, globals().get('_PUTARE_SETUP_STATE'))
    _expected_model = Path(MODEL_PATH) if MODEL_PATH else ROOT / 'models/gemma-4-26B-A4B-it-UD-IQ4_NL-lcb100-activity-ordered-e80.gguf'
    print('환경 준비 완료. 모델 경로:', _expected_model, '| 존재:', _expected_model.is_file())
    if not _expected_model.is_file():
        print('추론 전 모델 파일을 위 경로에 준비하세요.')
    _PUTARE_SETUP_READY = True


ninja가 없어 설치합니다.
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition, Compute Capability: 120
공식 LCB commit 28fef95ea8c9f7a547c8329f2cd3d32b92c1fa24, venv /content/Putare/6b061e47d30a-0a3dd9112ddf/.cache/putare-lcb/venv, source /content/Putare/6b061e47d30a-0a3dd9112ddf/.cache/putare-lcb/livecodebench/28fef95ea8c9f7a547c8329f2cd3d32b92c1fa24
환경 준비 완료. 모델 경로: /content/drive/MyDrive/models/gemma-4-26B-A4B-it-UD-IQ4_NL-lcb100-activity-ordered-e80.gguf | 존재: True


In [7]:
if RUN_GENERATION or RUN_GRADING:
    if not globals().get('_PUTARE_SETUP_READY', False):
        raise RuntimeError('먼저 SETUP_COLAB을 켜고 환경 준비를 완료하세요.')
    from pathlib import Path
    from tools import benchmark175
    MODEL = Path(MODEL_PATH)
    ENGINE = ROOT / '.cache/llama-build/bin/llama-expert-research'
    RUN_DIR = ROOT / 'runs/gemma4-e80-lcb175-001'
    if RUN_GENERATION:
        print(benchmark175.generate(RUN_DIR, model=MODEL, engine=ENGINE, checkpoint_dir=CHECKPOINT_DIR))
    if RUN_GRADING:
        print(benchmark175.grade(RUN_DIR, model=MODEL, engine=ENGINE, checkpoint_dir=CHECKPOINT_DIR))
else:
    print('생성 및 채점 대기 중입니다.')


1/175 livecodebench:3674
2/175 livecodebench:3677
3/175 livecodebench:3687
4/175 livecodebench:3692
5/175 livecodebench:3696
6/175 livecodebench:3697
7/175 livecodebench:3701
8/175 livecodebench:3704
9/175 livecodebench:3705
10/175 livecodebench:3708
11/175 livecodebench:3709
12/175 livecodebench:3714
13/175 livecodebench:3717
14/175 livecodebench:3720
15/175 livecodebench:3721
16/175 livecodebench:3722
17/175 livecodebench:3723
18/175 livecodebench:3725
19/175 livecodebench:3731
20/175 livecodebench:3733
21/175 livecodebench:3736
22/175 livecodebench:3737
23/175 livecodebench:3739
24/175 livecodebench:3743
25/175 livecodebench:3744
26/175 livecodebench:3747
27/175 livecodebench:3748
28/175 livecodebench:3750
29/175 livecodebench:3751
30/175 livecodebench:3753
31/175 livecodebench:3754
32/175 livecodebench:3759
33/175 livecodebench:3760
34/175 livecodebench:3762
35/175 livecodebench:3763
36/175 livecodebench:3764
37/175 livecodebench:3765
38/175 livecodebench:3768
39/175 livecodebench:

In [8]:
# 생성·공식 채점이 끝난 다음 전체 결과를 분할 보전합니다.
from pathlib import Path
import hashlib, json, zipfile, shutil

_run = ROOT / 'runs/gemma4-e80-lcb175-001'
_export = Path('/content/putare07-preserve-20261110')
if _export.exists():
    raise RuntimeError('보전 폴더가 이미 있습니다. 기존 결과를 덮어쓰지 않습니다: ' + str(_export))
_export.mkdir()
def _sha(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()
_files = sorted(p for p in _run.rglob('*') if p.is_file())
if any(p.is_symlink() or not p.resolve().is_relative_to(_run.resolve()) for p in _files):
    raise RuntimeError('결과 폴더에 안전하지 않은 경로가 있습니다.')
_inventory = {p.relative_to(_run).as_posix(): {'bytes': p.stat().st_size, 'sha256': _sha(p)} for p in _files}
_archive = _export / 'putare07-results.zip'
with zipfile.ZipFile(_archive, 'x', compression=zipfile.ZIP_STORED, allowZip64=True) as out:
    for p in _files:
        out.write(p, p.relative_to(_run).as_posix())
_parts = []
with _archive.open('rb') as inp:
    number = 1
    while True:
        block = inp.read(100 * 1024 * 1024)
        if not block:
            break
        part = _export / f'putare07-results.part-{number:04d}.bin'
        with part.open('xb') as out:
            out.write(block)
        _parts.append({'name': part.name, 'bytes': len(block), 'sha256': _sha(part)})
        number += 1
_manifest = {'run': str(_run), 'file_count': len(_inventory), 'files': _inventory,
             'archive': {'name': _archive.name, 'bytes': _archive.stat().st_size, 'sha256': _sha(_archive)}, 'parts': _parts}
_manifest_path = _export / 'manifest.json'
_manifest_path.write_text(json.dumps(_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
_drive = Path('/content/drive/MyDrive/Putare/results/07-e80-lcb175-20261110')
_drive.mkdir(parents=True, exist_ok=True)
for _name in ('manifest.json',):
    _target = _drive / _name
    if _target.exists():
        if _sha(_target) != _sha(_export / _name):
            raise RuntimeError('Drive의 기존 보전 파일과 충돌합니다.')
    else:
        shutil.copyfile(_export / _name, _target)
        if _sha(_target) != _sha(_export / _name):
            raise RuntimeError('Drive 보전 파일 해시 불일치')
print('PRESERVATION_READY', {'directory': str(_export), 'files': len(_inventory), 'parts': len(_parts), 'archive_bytes': _archive.stat().st_size, 'drive_manifest': str(_drive / 'manifest.json')})


PRESERVATION_READY {'directory': '/content/putare07-preserve-20261110', 'files': 1582, 'parts': 73, 'archive_bytes': 7645882945, 'drive_manifest': '/content/drive/MyDrive/Putare/results/07-e80-lcb175-20261110/manifest.json'}
